# 22 — Final Comparison: All 22 Algorithms

This notebook loads pre-saved results from `results/` (generated by `export_models.py`) and produces publication-quality comparison figures covering all 22 algorithms across all evaluation metrics.

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

BASE_DIR    = os.path.abspath(os.path.join(os.getcwd(), '..'))
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
FIGURES_DIR = os.path.join(os.getcwd(), 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 11, 'axes.titleweight': 'bold'})

COLORS = {'Classical ML': '#3498db', 'Deep Learning': '#e74c3c'}
print(f'Results dir: {RESULTS_DIR}')

## 1. Load Results

> **Run `export_models.py` first** to generate `results/all_results.csv`.

In [ ]:
csv_path = os.path.join(RESULTS_DIR, 'all_results.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f'Results file not found: {csv_path}\n'
        'Please run export_models.py first to generate it.'
    )
df = pd.read_csv(csv_path)
df = df.dropna(subset=['accuracy']).sort_values('accuracy', ascending=False).reset_index(drop=True)
print(f'Loaded {len(df)} models')
print(df[['model', 'category', 'accuracy', 'f1', 'auc', 'train_time']].to_string(index=False))

## 2. Summary Statistics

In [ ]:
print('\n=== By Category ===')
print(df.groupby('category')[['accuracy', 'f1', 'auc', 'train_time']].agg(['mean','max','min']).round(4).to_string())

print('\n=== Top 5 Models ===')
print(df.head(5)[['model', 'category', 'accuracy', 'f1', 'auc']].to_string(index=False))

## 3. Horizontal Bar Chart — Accuracy (All Models, Ranked)

In [ ]:
sorted_df = df.sort_values('accuracy', ascending=True)
colors = [COLORS.get(c, '#95a5a6') for c in sorted_df['category']]

fig, ax = plt.subplots(figsize=(12, max(8, len(sorted_df) * 0.42)))
bars = ax.barh(sorted_df['model'], sorted_df['accuracy'], color=colors, edgecolor='white', height=0.7)
ax.axvline(0.5, color='grey', linestyle='--', lw=1, alpha=0.6, label='Chance (0.5)')
ax.set_xlabel('Test Accuracy')
ax.set_title('All Algorithms — Accuracy Ranking')
ax.set_xlim(0, 1.1)
for bar, val in zip(bars, sorted_df['accuracy']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
patches = [mpatches.Patch(color=v, label=k) for k, v in COLORS.items()]
ax.legend(handles=patches, fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'comparison_accuracy_all.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Multi-Metric Comparison (Accuracy, F1, Precision, Recall)

In [ ]:
metrics_to_plot = [('accuracy', 'Accuracy'), ('f1', 'F1 Score'),
                   ('precision', 'Precision'), ('recall', 'Recall')]
fig, axes = plt.subplots(2, 2, figsize=(22, max(14, len(df) * 0.5)))
fig.suptitle('Fake News Detection — All Algorithms, All Metrics', fontsize=15, y=1.01)

for ax, (metric, title) in zip(axes.flatten(), metrics_to_plot):
    dm = df.dropna(subset=[metric]).sort_values(metric, ascending=True)
    clrs = [COLORS.get(c, '#95a5a6') for c in dm['category']]
    bars = ax.barh(dm['model'], dm[metric], color=clrs, edgecolor='white', height=0.7)
    ax.set_xlabel(title, fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.set_xlim(0, 1.08)
    ax.axvline(0.5, color='grey', linestyle='--', lw=0.7, alpha=0.6)
    for bar, val in zip(bars, dm[metric]):
        ax.text(val + 0.004, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=7)
    patches = [mpatches.Patch(color=v, label=k) for k, v in COLORS.items()]
    ax.legend(handles=patches, fontsize=8)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'comparison_all_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Top 15 Models — Grouped Bar Chart

In [ ]:
top15 = df.head(15)
x = np.arange(len(top15))
w = 0.2
fig, ax = plt.subplots(figsize=(16, 7))
ax.bar(x - w*1.5, top15['accuracy'],  w, label='Accuracy',  color='#3498db', alpha=0.85)
ax.bar(x - w*0.5, top15['f1'],        w, label='F1 Score',   color='#e74c3c', alpha=0.85)
ax.bar(x + w*0.5, top15['precision'], w, label='Precision',  color='#2ecc71', alpha=0.85)
ax.bar(x + w*1.5, top15['recall'],    w, label='Recall',     color='#f39c12', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top15['model'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Top 15 Models — All Metrics Side-by-Side')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'comparison_top15_grouped.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. AUC-ROC Comparison

In [ ]:
da = df.dropna(subset=['auc']).sort_values('auc', ascending=True)
colors_a = [COLORS.get(c, '#95a5a6') for c in da['category']]

fig, ax = plt.subplots(figsize=(12, max(6, len(da) * 0.42)))
bars = ax.barh(da['model'], da['auc'], color=colors_a, edgecolor='white', height=0.7)
ax.axvline(0.5, color='red', linestyle='--', lw=1.2, label='Random (AUC=0.5)')
ax.set_xlabel('AUC-ROC')
ax.set_title('AUC-ROC Score — All Algorithms')
ax.set_xlim(0, 1.08)
for bar, val in zip(bars, da['auc']):
    ax.text(val + 0.004, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
patches = [mpatches.Patch(color=v, label=k) for k, v in COLORS.items()]
ax.legend(handles=patches + [plt.Line2D([0],[0], color='red', linestyle='--', label='Random (0.5)')],
          fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'comparison_auc_all.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Training Time Comparison

In [ ]:
dt = df.dropna(subset=['train_time']).sort_values('train_time', ascending=True)
colors_t = [COLORS.get(c, '#95a5a6') for c in dt['category']]

fig, ax = plt.subplots(figsize=(12, max(6, len(dt) * 0.42)))
bars = ax.barh(dt['model'], dt['train_time'], color=colors_t, edgecolor='white', height=0.7)
ax.set_xlabel('Training Time (seconds)')
ax.set_title('Training Time — All Algorithms')
for i, (val, bar) in enumerate(zip(dt['train_time'], bars)):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}s', va='center', fontsize=8)
patches = [mpatches.Patch(color=v, label=k) for k, v in COLORS.items()]
ax.legend(handles=patches, fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'comparison_training_time.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Accuracy vs Training Time (Efficiency Scatter)

In [ ]:
scatter_df = df.dropna(subset=['accuracy', 'train_time'])
fig, ax = plt.subplots(figsize=(11, 7))
for cat, color in COLORS.items():
    sub = scatter_df[scatter_df['category'] == cat]
    ax.scatter(sub['train_time'], sub['accuracy'], c=color, s=80,
               label=cat, alpha=0.85, edgecolors='white', linewidths=0.8)
    for _, row in sub.iterrows():
        ax.annotate(row['model'], (row['train_time'], row['accuracy']),
                    textcoords='offset points', xytext=(4, 4), fontsize=7, alpha=0.8)
ax.set_xscale('log')
ax.set_xlabel('Training Time (seconds, log scale)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Accuracy vs Training Time — Efficiency Trade-off')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'comparison_accuracy_vs_time.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Heatmap: All Models × All Metrics

In [ ]:
metric_cols = ['accuracy', 'precision', 'recall', 'f1', 'auc']
heat_df = df.set_index('model')[metric_cols].dropna()

fig, ax = plt.subplots(figsize=(10, max(8, len(heat_df) * 0.38)))
sns.heatmap(heat_df, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.5, vmax=1.0, ax=ax,
            linewidths=0.4, cbar_kws={'label': 'Score', 'shrink': 0.7})
ax.set_title('All Models — Metric Heatmap', fontsize=14)
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'comparison_metric_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Classical ML vs Deep Learning — Box Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, ['accuracy', 'f1', 'auc']):
    data = [df[df.category == cat][metric].dropna().values
            for cat in ['Classical ML', 'Deep Learning']]
    bp = ax.boxplot(data, labels=['Classical ML', 'Deep Learning'],
                    patch_artist=True, notch=False, widths=0.5)
    for patch, color in zip(bp['boxes'], ['#3498db', '#e74c3c']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_ylabel(metric.title())
    ax.set_title(f'{metric.upper()} Distribution')
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Classical ML vs Deep Learning — Score Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'comparison_category_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Final Ranked Table

In [ ]:
display_df = df[['model','category','accuracy','f1','precision','recall','auc','train_time']].copy()
display_df.columns = ['Model','Category','Accuracy','F1','Precision','Recall','AUC-ROC','Time(s)']
display_df.index = range(1, len(display_df) + 1)
display_df.index.name = 'Rank'
print('\n' + '='*95)
print('  FINAL RESULTS — ALL 22 ALGORITHMS RANKED BY ACCURACY')
print('='*95)
print(display_df.round(4).to_string())
print('='*95)